# Compute Forward Operator

# Batch Wise Run

Run the code for all subjects of a given dataset.

## 📦 Imports

In [1]:
import mindmap as mm
from pathlib import Path
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- `hmodel`: Head conductor model
- `src_raw`: Source space (before computing forward).
- `eeg`: EEG
- `trans`: Transformation for EEG coregistration

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.

🛠️ **<font color=violet>Other options</font>**:
- `overwrite`: If True overwrite old files. If False, skip subjects already having files.

In [2]:
############# Parameters ###############
# configuration
fconfig = "../configurations/spike.yml"

# other options 
overwrite = True
########################################

# configuration
config   = mm.io.read_config(fconfig)
subjects = config["SUBS"]
mm_dir   = Path(config["MM_DIR"])

ffwd = lambda sub: mm_dir/sub/"fwd"/f"{sub}_fwd.fif"

FileNotFoundError: [Errno 2] No such file or directory: '../configurations/spike.yml'

## ♾️ Loop

📤 **<font color=pink>Outputs</font>** (files in freesurfer reconstruction bem folder)
- `fwd`: Forward operator

In [ ]:
for sub in subjects:
    # check file existence
    fname = ffwd(sub)

    if fname.exists() and not overwrite:
        print(f"\033[93mForward solution already exists for subject {sub}\033[0m")
        continue 
    if not fname.parent.exists():
        fname.parent.mkdir(parents=True)
    
    try: 
         # data
        data = mm.io.read_dataset(sub, "hmodel", "src_raw", "eeg", "trans")
        
        # compute forward solution
        forward = mne.make_forward_solution(
            info=data["eeg"].info,
            trans=data["trans"],
            src=data["src_raw"],
            bem=data["hmodel"],
            meg=False,
            eeg=True,
            verbose="warning",
        )
    except Exception as e: 
        print(f"\033[91mFailed to compute forward solution for subject {sub} due to: {e}\033[0m")
        continue

    # save 
    mne.write_forward_solution(fname, forward, overwrite=overwrite, verbose="warning")
    print(f"\033[92mSuccessfully saved forward solution for subject {sub}\033[0m")

Preprocessed EEG not found, reading raw instead.
Failed to compute forward solution for subject sub-pat01 due to: File does not exist: "/media/nino/DATA/SPIKE/derivatives/mindmap/sub-pat01/eeg/sub-pat01_eeg-trans.fif"
Preprocessed EEG not found, reading raw instead.
Failed to compute forward solution for subject sub-pat02 due to: File does not exist: "/media/nino/DATA/SPIKE/derivatives/mindmap/sub-pat02/src/sub-pat02_src.fif"
Preprocessed EEG not found, reading raw instead.
Failed to compute forward solution for subject sub-pat03 due to: File does not exist: "/media/nino/DATA/SPIKE/derivatives/mindmap/sub-pat03/src/sub-pat03_src.fif"
Preprocessed EEG not found, reading raw instead.
Failed to compute forward solution for subject sub-pat04 due to: File does not exist: "/media/nino/DATA/SPIKE/derivatives/mindmap/sub-pat04/src/sub-pat04_src.fif"
Preprocessed EEG not found, reading raw instead.
Failed to compute forward solution for subject sub-pat05 due to: File does not exist: "/media/nin

# Step Wise Run

Run the code step by step for a single subject.

## 📦 Imports

In [1]:
import mindmap as mm 
import mne
import pyvista as pv
from pathlib import Path

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- `hmodel`: Head conductor model.
- `src_raw`: Source space (before computing forward).
- `eeg`: EEG.
- `trans`: Affine transformation for EEG coregistration.

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)
- `hmodel`: Type of head model (`hmodel` for solving inverse problem or `hmodel-sim` for generating simulations).

In [2]:
############## Parameters ###############
fconfig = "../../papers/simf/configuration.yml"
sub     = 1
hmodel  = "hmodel"
#########################################

reader = mm.io.dataset_reader(fconfig)
mm_dir = reader.config["MM_DIR"]

# data
data = reader(sub, hmodel, "src500-raw", "eeg", "trans")
data["src-raw"] = data["src500-raw"]

Coregistered EEG found
DATA SUMMARY
Dataset: simf
Subject: vepcon_sub-10
Retrieved objects:
- src500-raw
- hmodel
- eeg
- trans
Missing objects:


🌟 **Check coregistration**

In [3]:
head = data[hmodel]["surfs"][0]
sens = data["eeg"]._get_channel_positions()
sens = mne.transforms.apply_trans(data["trans"], sens)

# plot
head = mm.utils.surf_to_polydata(head["rr"], head["tris"])
sens = pv.PolyData(sens)

p = pv.Plotter()
p.add_mesh(head, color="#D2A18C", smooth_shading=False, label="head")
p.add_mesh(sens, render_points_as_spheres=True, color="black", label="sensors", point_size=8)

p.add_legend(face="circle")
p.show()

Widget(value='<iframe src="http://localhost:45343/index.html?ui=P_0x7f985e72a210_0&reconnect=auto" class="pyvi…

## Compute Forward Operator

In [4]:
forward = mne.make_forward_solution(
    info    = data["eeg"].info,
    trans   = data["trans"],
    src     = data["src-raw"],
    bem     = data[hmodel],
    meg     = False,
    eeg     = True,
    verbose = "warning",
)

🌟 **Visualize Forward solution**

In [5]:
mm.viz.plot_leadfield(data["eeg"], forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:45343/index.html?ui=P_0x7f9856f5cb90_1&reconnect=auto" class="pyvi…

💾 **Save Forward Solution** <br>
🛠️ **<font color=violet>Options</font>**:
- `overwrite`: Overwrite existing forward solution.


In [ ]:
############# Parameters #############
overwrite = True
######################################

if hmodel == "hmodel":
    ffwd = mm_dir/data["sub"]/"fwd"/f"{data['sub']}_fwd.fif"
else:
    ffwd = mm_dir/data["sub"]/"fwd"/f"{data['sub']}_sim_fwd.fif"

if not ffwd.parent.exists():
    ffwd.parent.mkdir(parents=True)

mne.write_forward_solution(ffwd, forward, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved forward solution for subject {data['sub']} with model '{hmodel}'\033[0m")

Successfully saved forward solution for subject vepcon_sub-10 with model 'hmodel'
